In [5]:
import numpy as np
import re, pandas as pd
import os
BASE_DIR = '/home/users/yyhuang/ICSD/deduplicate/job_script/ICSD_filter/formal_run'
results = pd.read_pickle(os.path.join(BASE_DIR, 'ICSD_valid_dedup_Stol_1e-4_vtol_1e-2_occ_tol_1.05.pkl'))
intersect_orb_sites = pd.read_csv(os.path.join(BASE_DIR, 'intersect_orb_sites.csv'))   # records of intersecting wyckoff orbit/sites in positional disorder
intersect_orb_err = pd.read_csv(os.path.join(BASE_DIR, 'intersect_orb_errors.json'))   # records of error when processing positional disorder
occ_err_sites = pd.read_csv(os.path.join(BASE_DIR, 'occ_err_sites.csv'))               # records of wyckoff orbit/sites that exceed occupancy tolerance (defualt 1.05)
equivalent_but_distinct_sites = pd.read_csv(os.path.join(BASE_DIR, 'equivalent_but_distinct_sites.csv')) # records of wyckoff sites that
same_valence_sites = pd.read_csv(os.path.join(BASE_DIR, 'same_valence_sites.csv'))     #records of wyckoff sites that shares same elements with same oxidation number
wyck_float_warn = pd.read_csv(os.path.join(BASE_DIR, 'wyck_float_warn.csv'))           #records of wyckoff sites that generate number coordinates > multiplicity due to floating-point error

In [2]:
#
unwanted_codes = set().union(*[
    wyck_float_warn['CollectionCode'].unique().tolist(), 
    #intersect_orb_sites['CollectionCode'].unique().tolist(),   #untag this to remove positional disorder
    occ_err_sites['CollectionCode'].unique().tolist(), 
    equivalent_but_distinct_sites['CollectionCode'].unique().tolist(), 
    ])
ICSD_screened = results[~results['CollectionCode'].isin(unwanted_codes)]
display(len(unwanted_codes),len(ICSD_screened))

1462

204481

In [6]:
pd.set_option('display.max_colwidth', None)
ICSD_group = (
    ICSD_screened
    .groupby(['disorder_label'], as_index=False)
    .agg(
        n_total            = ('disorder_label', 'size'),
        CollectionCode_list= ('CollectionCode', list),
        StructuredFormula_list=('StructuredFormula', list),
        WyckoffSequence_lists = ('WyckoffSequence',list),
        #disorder_lists = ('disordered_list',list),
        degree_of_mixing_list=('degree_of_mixing', list),
        n_disorder         = ('is_disorder', 'sum'),
    )
)
ICSD_group['n_disorder'] = ICSD_group['n_disorder'].astype(int)
entries_unique = len(ICSD_group.query('n_total == 1'))
entries_with_duplicate = len(ICSD_screened)-entries_unique
unique_materials = len(ICSD_group)
duplicate_materials = len(ICSD_screened)-len(ICSD_group)
display(f"Unique structure groups: {unique_materials}")
display(f"Unique disorder structures groups: {len(ICSD_group.query('n_disorder >=1'))}")
display(f"single-entry groups: {entries_unique}")
pd.set_option('display.max_colwidth', 20)

'Unique structure groups: 115459'

'Unique disorder structures groups: 57686'

'single-entry groups: 86867'

In [ ]:
#deduplicate by selecting CollectionCode with highest absolute value of degree of mixing (would give only one representative for each structure group)
# def dedupe_by_highest_DOM(deg_list, code_list):
#     if deg_list is None or len(deg_list) == 0:
#         return code_list[0] if code_list else np.nan

#     arr = np.array(deg_list, dtype=float)

#     if np.all(np.isnan(arr)):
#         return code_list[0] if code_list else np.nan

#     idx = int(np.nanargmax(np.abs(arr)))

#     if code_list is None or idx >= len(code_list):
#         return np.nan
#     return code_list[idx]

# ICSD_group['selected_collection_code'] = ICSD_group.apply(
#     lambda r: dedupe_by_highest_DOM(r['degree_of_mixing_list'], r['CollectionCode_list']),
#     axis=1
# )

In [ ]:
# #deduplicate by removing CollectionCode with same degree of mixing (would give multiple representative with differnt DOM for each structure group)
# def dedupe_by_same_DOM(deg_list, code_list):
#     if deg_list is None or code_list is None:
#         return code_list or []
#     out = []
#     seen = set()
#     for d, c in zip(deg_list, code_list):
#         key = ('__nan__' if pd.isna(d) else float(d))
#         if key in seen:
#             continue
#         seen.add(key)
#         out.append(c)
#     return out

# ICSD_group['selected_collection_code'] = ICSD_group.apply(
#     lambda r: dedupe_by_same_DOM(r['degree_of_mixing_list'], r['CollectionCode_list']),
#     axis=1
# )

In [ ]:
#test selection
pd.set_option('display.max_colwidth', None)
display(ICSD_group[ICSD_group['disorder_label'] == 'a_c_b_166_Li_O_{Co+Mn+Ni}'])
pd.set_option('display.max_colwidth', 20)

,disorder_label,n_total,CollectionCode_list,StructuredFormula_list,WyckoffSequence_lists,degree_of_mixing_list,n_disorder,selected_collection_code
19286,a_c_b_166_Li_O_{Co+Mn+Ni},13,"[5302, 15754, 27874, 38734, 61920, 61921, 99891, 145745, 162291, 191531, 191533, 259696, 259697]","[Li (Ni0.58 Co0.25 Mn0.17) O2, Li (Ni0.58 Co0.25 Mn0.17) O2, Li (Ni0.58 Co0.25 Mn0.17) O2, Li Ni0.6 Co0.2 Mn0.2 O2, Li Ni0.6 Co0.2 Mn0.2 O2, Li Ni0.6 Co0.2 Mn0.2 O2, Li ((Ni0.25 Co0.5 Mn0.25) O2), Li Ni0.8 Mn0.1 Co0.1 O2, Li (Co0.8 Ni0.1 Mn0.1) O2, Li (Mn0.375 Ni0.375 Co0.25) O2, Li (Mn0.375 Ni0.375 Co0.25) O2, Li (Ni0.333 Co0.333 Mn0.333) O2, Li (Ni0.333 Co0.333 Mn0.333) O2]","[c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a, c b a]","[-0.8772, -0.8772, -0.8772, -0.865, -0.865, -0.865, 0.9464, -0.5817, 0.5817, -0.9851, -0.9851, 0.9999, 0.9999]",13,"[5302, 38734, 99891, 145745, 162291, 191531, 259696]"


In [21]:
duplicated_codes = {c for v in ICSD_group['selected_collection_code'].dropna() for c in (v if isinstance(v, (list,tuple,set)) else [v])}
ICSD_deduplicated = ICSD_screened[ICSD_screened['CollectionCode'].isin(duplicated_codes)].copy()
display(f"removed duplicated entries, {len(ICSD_deduplicated)} left")

'removed duplicated entries, 143477 left'